# Day 2 Lab: Evidence Discipline and Language Models in Practice

Bias-variance, survivorship bias, walk-forward validation, strategy
evaluation, and LLM architecture

> **Expected Time**
>
> -   Exercise 1 (Bias-variance and bootstrap): ≈ 25 minutes
> -   Exercise 2 (Survivorship bias): ≈ 25 minutes
> -   Exercise 3 (Walk-forward strategy evaluation): ≈ 40 minutes
> -   Exercise 4 (Evidence quality assessment): ≈ 20 minutes
> -   Exercise 5 (LLM bridge — interactive critique): ≈ 20 minutes
> -   Exercise 6 (Tokenisation, embeddings, and attention): ≈ 28 minutes
> -   Total: ≈ 158 minutes
>
> **In-session:** Exercises 3 (Lab A, 35 min) and 6 (Lab B, 15 min). The
> rest are for independent work before or after Day 3.

<figure>
<a
href="https://colab.research.google.com/github/quinfer/minimba-notebooks/blob/main/lab02_evidence_discipline.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Setup (Colab-only installs)

In [ ]:
# Run this cell first — installs everything needed for all exercises
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import stats
    print("Core packages: OK")
except ImportError:
    install("numpy pandas matplotlib scipy statsmodels")
    import numpy as np, pandas as pd, matplotlib.pyplot as plt
    from scipy import stats

# Exercise 6 dependencies (NLP packages — install once, used in Ex 6)
try:
    from transformers import AutoTokenizer
    from sentence_transformers import SentenceTransformer
    print("NLP packages: OK")
except ImportError:
    print("Installing NLP packages (one-time, ~2 min in Colab)...")
    for pkg in ["transformers", "sentence-transformers", "bertviz"]:
        install(pkg)
    print("NLP packages installed.")

print("Setup complete.")

## Before You Start

This lab makes Day 2’s concepts concrete. Each exercise corresponds to a
key lesson from the lectures:

| Exercise | Day 2 concept | What you’ll do |
|-------------------|-------------------------|-----------------------------|
| 1 | Bias-variance tradeoff | See overfitting happen in real time |
| 2 | Survivorship bias | Quantify how excluding failed funds distorts returns |
| 3 | Walk-forward validation | Evaluate a simple strategy honestly |
| 4 | Evidence quality | Assess a real-world FinTech claim |
| 5 | LLM bridge | Use an LLM to critique a claim with the Day 2 framework (interactive) |
| 6 | Tokenisation, embeddings, attention | See how LLMs process financial text and where the architecture creates failure modes |

All code is provided. Your job is to **run it, change parameters,
interpret results, and write short reflections** connecting observations
to theory. Where you see **Try it** callouts, change the suggested
parameters and re-run the cell to see how results change — this makes
the lab interactive.

# Exercise 1 — The Bias-Variance Tradeoff (25 min)

## Why This Matters

Every model balances two errors: **bias** (too simple, misses the
pattern) and **variance** (too complex, fits the noise). In finance,
noise dominates signal, so overfitting is the default failure mode. This
exercise lets you see it happen.

## Task 1a: Fit Polynomials of Increasing Complexity

We generate data from a known function (sine wave) with noise, then fit
polynomials of different degrees. Watch what happens as complexity
increases.

In [ ]:
np.random.seed(42)
n = 30
x = np.linspace(0, 1, n)
y_true = np.sin(2 * np.pi * x)
y = y_true + np.random.normal(0, 0.3, size=n)
x_fine = np.linspace(0, 1, 200)

degrees = [1, 3, 5, 15, 25]
fig, axes = plt.subplots(1, len(degrees), figsize=(16, 3.5))

for ax, d in zip(axes, degrees):
    coef = np.polyfit(x, y, d)
    y_pred = np.polyval(coef, x_fine)
    ax.scatter(x, y, alpha=0.5, s=20, color='steelblue')
    ax.plot(x_fine, np.clip(y_pred, -2.5, 2.5), 'r-', linewidth=2)
    ax.plot(x_fine, np.sin(2 * np.pi * x_fine), 'g--', alpha=0.4, linewidth=1)
    ax.set_title(f'Degree {d}')
    ax.set_ylim(-2.5, 2.5)
    ax.grid(alpha=0.2)

plt.suptitle('Increasing Model Complexity: From Underfit to Overfit', fontsize=13)
plt.tight_layout()
plt.show()

> **Try it (interactive)**
>
> Change `degrees = [1, 3, 5, 15, 25]` to other values (e.g. add 10 or
> 30) or increase the noise in `np.random.normal(0, 0.3, ...)` to 0.5,
> then re-run. Observe how overfitting appears earlier when noise is
> higher.

### Questions (Write 100–150 words)

1.  At which degree does the fit look “best”? What happens at degree 25?
2.  If you only had the training data (blue dots) and no knowledge of
    the true function (green dashed), how would you choose the right
    complexity?
3.  Connect this to finance: if the “true function” is a very weak
    signal (R² ~ 1–2%), what does overfitting look like?

## Task 1b: Bootstrap Confidence Interval

The bootstrap gives us a way to quantify uncertainty without assuming a
distribution. We resample our data many times and compute the statistic
of interest on each resample.

In [ ]:
np.random.seed(42)
# Simulate 250 daily returns (1 year)
returns = np.random.normal(0.0003, 0.015, size=250)

# Bootstrap the mean
n_boot = 5000
boot_means = np.array([np.mean(np.random.choice(returns, size=len(returns), replace=True))
                        for _ in range(n_boot)])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
sample_mean = returns.mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(boot_means, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(sample_mean, color='red', linewidth=2, label=f'Sample mean: {sample_mean:.5f}')
ax.axvline(ci_low, color='orange', linestyle='--', linewidth=2, label=f'95% CI: [{ci_low:.5f}, {ci_high:.5f}]')
ax.axvline(ci_high, color='orange', linestyle='--', linewidth=2)
ax.axvline(0, color='black', linewidth=1, linestyle=':')
ax.set_title('Bootstrap Distribution of Mean Daily Return')
ax.set_xlabel('Mean daily return')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Sample mean: {sample_mean:.6f}")
print(f"95% Bootstrap CI: [{ci_low:.6f}, {ci_high:.6f}]")
print(f"Does the CI include zero? {'Yes — not significantly different from zero' if ci_low < 0 < ci_high else 'No — significantly different from zero'}")

> **Try it (interactive)**
>
> Change the sample size (e.g. `size=500` or `size=100` in the `returns`
> line) or the drift in `np.random.normal(0.0003, ...)` (e.g. to
> `0.001`), then re-run. See how the CI width and the “includes zero?”
> conclusion change with sample size and signal strength.

### Key Insight

Even with a positive sample mean, the confidence interval often includes
zero. With daily financial data, one year of observations is frequently
**not enough** to distinguish a genuine signal from noise. This is the
signal-to-noise problem in action.

> **Optional note (instructors)**
>
> With the default seed, the sample mean is close to the true drift
> (0.0003); the bootstrap 95% CI spans roughly ±2× the standard error of
> the mean. The fact that the CI includes zero is by design — it
> reflects the high noise in daily returns — so the “often includes
> zero” message is the intended lesson.

# Exercise 2 — Survivorship Bias (25 min)

## Why This Matters

When you study “the market” or “mutual funds” or “hedge funds,” you
typically analyse assets that survived to the present day. Failed funds,
delisted stocks, and bankrupt companies disappear from the data. This
systematically inflates apparent returns.

## Task 2a: Simulate 100 Funds with Failures

In [ ]:
np.random.seed(42)
n_funds = 100
n_months = 120  # 10 years
failure_threshold = -0.30  # Fund closes after 30% cumulative drawdown

# All funds have IDENTICAL expected returns (no skill)
all_returns = np.random.normal(0.005, 0.04, size=(n_months, n_funds))
cumulative = np.cumprod(1 + all_returns, axis=0)

# Track survival
alive = np.ones(n_funds, dtype=bool)
death_month = np.full(n_funds, n_months)
for f in range(n_funds):
    for t in range(1, n_months):
        if cumulative[t, f] < (1 + failure_threshold) * cumulative[0, f]:
            alive[f] = False
            death_month[f] = t
            cumulative[t+1:, f] = np.nan
            all_returns[t+1:, f] = np.nan
            break

n_survivors = alive.sum()
n_failed = n_funds - n_survivors

# Average returns: all funds vs survivors only
all_mean = np.nanmean(all_returns, axis=1)
survivor_mask = all_returns.copy()
survivor_mask[:, ~alive] = np.nan
survivor_mean = np.nanmean(survivor_mask, axis=1)

cum_all = np.cumprod(1 + all_mean)
cum_survivor = np.cumprod(1 + np.nan_to_num(survivor_mean, nan=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Individual fund paths
for f in range(n_funds):
    colour = 'coral' if not alive[f] else 'steelblue'
    alpha = 0.15
    axes[0].plot(cumulative[:, f], color=colour, alpha=alpha, linewidth=0.5)
axes[0].set_title(f'{n_funds} Funds: {n_survivors} Survived (blue), {n_failed} Failed (red)')
axes[0].set_ylabel('Cumulative Growth (£1)')
axes[0].set_xlabel('Month')
axes[0].grid(alpha=0.3)

# Panel 2: Average comparison
axes[1].plot(cum_all, linewidth=2.5, label=f'All {n_funds} funds', color='steelblue')
axes[1].plot(cum_survivor, linewidth=2.5, label=f'Survivors only ({n_survivors})', color='coral')
axes[1].set_title('Survivorship Bias: Survivors Look Better')
axes[1].set_ylabel('Cumulative Growth (£1)')
axes[1].set_xlabel('Month')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

bias = (cum_survivor[-1] - cum_all[-1]) / cum_all[-1] * 100
print(f"Survivors: {n_survivors} / {n_funds}")
print(f"Final value (all funds): £{cum_all[-1]:.2f}")
print(f"Final value (survivors): £{cum_survivor[-1]:.2f}")
print(f"Survivorship bias: {bias:.1f}% overstated")

### Questions (Write 150–200 words)

1.  All 100 funds have **identical** expected returns (no skill). Yet
    the survivors appear to outperform. Why?
2.  A financial advisor shows you that “the average UK equity fund
    returned 8% annually over the last decade.” What question should you
    ask?
3.  How would this bias affect your evaluation of a FinTech lending
    platform that reports its “historical default rate” using only
    currently active loans?

## Task 2b: Change the Parameters

Try changing these parameters and observe the effect on survivorship
bias:

-   `failure_threshold = -0.20` (more sensitive trigger — more funds
    fail)
-   `failure_threshold = -0.50` (more robust — fewer fail)
-   `n_months = 60` (5 years instead of 10)
-   `np.random.seed(123)` (different random draw)

Write one sentence on which parameter had the biggest effect on bias
magnitude.

# Exercise 3 — Walk-Forward Strategy Evaluation (40 min)

## Why This Matters

This is the core practical exercise. You will evaluate a simple
moving-average crossover strategy using walk-forward validation and
honest baselines — the same discipline that separates genuine evidence
from backtest mining.

## Task 3a: Load Data and Implement Strategy

In [ ]:
np.random.seed(42)
# Simulate daily returns for a stock (or load Bloomberg data if available)
n_days = 1260  # ~5 years
dates = pd.date_range('2019-01-01', periods=n_days, freq='B')
daily_returns = pd.Series(np.random.normal(0.0003, 0.015, size=n_days), index=dates, name='returns')

# Load Bloomberg data: local path when available, else public URL (e.g. Colab)
from pathlib import Path
bloomberg_path = Path("resources/bloomberg_database.parquet")
bloomberg_url = "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/main/resources/bloomberg_database.parquet"
source = bloomberg_path if bloomberg_path.exists() else bloomberg_url
try:
    df = pd.read_parquet(source)
    spy = df[df['ticker'] == 'SPY'].set_index('date')['PX_LAST'].sort_index()
    daily_returns = spy.pct_change().dropna()
    daily_returns.name = 'returns'
    print(f"Loaded Bloomberg SPY data: {len(daily_returns)} observations")
except Exception:
    print(f"Using simulated data: {len(daily_returns)} observations")

# Compute moving averages
prices = (1 + daily_returns).cumprod()
ma_short = prices.rolling(20).mean()
ma_long = prices.rolling(60).mean()

# Generate signal: 1 when short MA > long MA, 0 otherwise
signal = (ma_short > ma_long).astype(int).shift(1).dropna()  # shift(1) = no look-ahead

# Strategy returns
strategy_returns = signal * daily_returns.loc[signal.index]
buyhold_returns = daily_returns.loc[signal.index]

# Cumulative comparison
cum_strategy = (1 + strategy_returns).cumprod()
cum_buyhold = (1 + buyhold_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_buyhold, label='Buy & Hold (baseline)', linewidth=2, color='steelblue')
ax.plot(cum_strategy, label='MA Crossover (20/60)', linewidth=2, color='coral')
ax.set_title('Strategy vs Buy-and-Hold: Full Sample (Not Walk-Forward Yet)')
ax.set_ylabel('Cumulative Growth (£1)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Performance metrics
sharpe_strategy = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
sharpe_buyhold = buyhold_returns.mean() / buyhold_returns.std() * np.sqrt(252)
print(f"Strategy Sharpe (full sample): {sharpe_strategy:.2f}")
print(f"Buy & Hold Sharpe (full sample): {sharpe_buyhold:.2f}")

> **Warning**
>
> This is a **full-sample** backtest. It does NOT use walk-forward
> validation. The results may be misleading. We fix this next.

## Task 3b: Walk-Forward Evaluation

Now let’s evaluate the strategy properly using expanding-window
walk-forward validation.

In [ ]:
# Walk-forward parameters
min_train = 252  # Minimum 1 year of training data
test_window = 63  # Test on 3 months at a time

n = len(daily_returns)
fold_results = []
all_wf_strategy = []
all_wf_buyhold = []

start = min_train
while start + test_window <= n:
    # Training data: everything up to 'start'
    train_returns = daily_returns.iloc[:start]
    train_prices = (1 + train_returns).cumprod()
    
    # Test data: next 'test_window' days
    test_slice = slice(start, start + test_window)
    test_returns = daily_returns.iloc[test_slice]
    test_prices = (1 + daily_returns.iloc[:start + test_window]).cumprod()
    
    # Compute MAs using only data up to and including current day
    ma_s = test_prices.rolling(20).mean()
    ma_l = test_prices.rolling(60).mean()
    sig = (ma_s > ma_l).astype(int).shift(1)
    
    # Strategy returns in test window
    test_sig = sig.iloc[test_slice]
    strat_ret = test_sig * test_returns
    
    sharpe_wf = strat_ret.mean() / strat_ret.std() * np.sqrt(252) if strat_ret.std() > 0 else 0
    sharpe_bh = test_returns.mean() / test_returns.std() * np.sqrt(252) if test_returns.std() > 0 else 0
    
    fold_results.append({
        'fold': len(fold_results) + 1,
        'test_start': daily_returns.index[start],
        'sharpe_strategy': sharpe_wf,
        'sharpe_buyhold': sharpe_bh,
        'strategy_beats_bh': sharpe_wf > sharpe_bh,
    })
    
    all_wf_strategy.extend(strat_ret.tolist())
    all_wf_buyhold.extend(test_returns.tolist())
    
    start += test_window

results_df = pd.DataFrame(fold_results)

# Overall walk-forward performance
wf_strat = np.array(all_wf_strategy)
wf_bh = np.array(all_wf_buyhold)
wf_sharpe_strat = np.nanmean(wf_strat) / np.nanstd(wf_strat) * np.sqrt(252)
wf_sharpe_bh = np.nanmean(wf_bh) / np.nanstd(wf_bh) * np.sqrt(252)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Sharpe ratio by fold
axes[0].bar(results_df['fold'] - 0.15, results_df['sharpe_strategy'], width=0.3,
            label='Strategy', color='coral', alpha=0.8)
axes[0].bar(results_df['fold'] + 0.15, results_df['sharpe_buyhold'], width=0.3,
            label='Buy & Hold', color='steelblue', alpha=0.8)
axes[0].set_xlabel('Walk-Forward Fold')
axes[0].set_ylabel('Annualised Sharpe Ratio')
axes[0].set_title('Walk-Forward Sharpe by Fold')
axes[0].legend()
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].grid(alpha=0.3)

# Panel 2: Cumulative walk-forward P&L
axes[1].plot(np.cumprod(1 + np.array(all_wf_buyhold)), label='Buy & Hold', color='steelblue', linewidth=2)
axes[1].plot(np.cumprod(1 + np.array(all_wf_strategy)), label='Strategy', color='coral', linewidth=2)
axes[1].set_title('Cumulative Walk-Forward Performance')
axes[1].set_ylabel('Growth of £1')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

win_rate = results_df['strategy_beats_bh'].mean()
print(f"\nWalk-forward results:")
print(f"  Strategy Sharpe (aggregate): {wf_sharpe_strat:.2f}")
print(f"  Buy & Hold Sharpe (aggregate): {wf_sharpe_bh:.2f}")
print(f"  Strategy beats B&H in {win_rate:.0%} of folds")
print(f"  Number of folds: {len(results_df)}")
print(f"\nFull-sample Sharpe was {sharpe_strategy:.2f} — walk-forward is {wf_sharpe_strat:.2f}")
print(f"This {'supports' if wf_sharpe_strat > wf_sharpe_bh else 'does not support'} the strategy.")

Even when full-sample and walk-forward Sharpe are similar for the
strategy, the comparison to the baseline (buy-and-hold) can still show
that the strategy does not add value — so the baseline is as important
as the validation method.

### Questions (Write 200–300 words)

1.  Compare the full-sample Sharpe ratio to the walk-forward Sharpe
    ratio. Why are they different?
2.  Does the strategy consistently beat buy-and-hold across folds, or is
    performance concentrated in specific periods?
3.  Would you recommend deploying this strategy with real money? What
    additional information would you need?
4.  You tested one parameter combination (20/60 days). If you also
    tested 10/30, 20/50, 30/90, and 50/200, how would this affect your
    confidence in the result? (Hint: multiple testing.)

# Exercise 4 — Evidence Quality Assessment (20 min)

## No Code Required

Choose **one** of the following real-world claims and write a 200–300
word assessment using the frameworks from today:

**Claim A:** “Our robo-advisor has outperformed the market by 2%
annually over the last 5 years.”

**Claim B:** “Our AI credit scoring model reduces default rates by 25%
compared to traditional methods.”

**Claim C:** “Our algorithmic trading system has achieved a Sharpe ratio
of 2.5 since launch.”

**For your chosen claim, address:**

1.  **Pitfall check:** Which of the five pitfalls might apply?
    (Significance vs importance, p-hacking, publication bias, etc.)
2.  **Bias check:** Could survivorship bias, look-ahead bias, or
    selection bias explain the result?
3.  **Baseline check:** What is the relevant baseline? Does the claim
    beat it meaningfully?
4.  **Overfitting check:** Where on the overfitting spectrum does this
    claim sit?
5.  **What would you ask?** List 3 specific questions you would ask the
    vendor.

> **Connection to Assessment**
>
> This exercise is direct preparation for the final briefing note. The
> assessment asks you to evaluate a real FinTech or AI claim using
> exactly these frameworks.

# Exercise 5 — LLM Bridge: Interactive Critique (20 min)

## Why This Matters

Day 2’s evidence discipline applies **unchanged** to LLM and AI
benchmark claims. This exercise is interactive: you use a structured
prompt with any LLM (e.g. ChatGPT, Claude, or a Colab notebook with an
API) to critique a vendor-style claim using the five pitfalls and bias
checklist. You see the framework in action and can reuse the prompt on
real claims.

## Task 5a: Prompt Template (Copy and Use)

Copy the following prompt into your preferred LLM. Replace
`[PASTE CLAIM HERE]` with one of the claims below (or your own).

``` text
You are an evidence-quality reviewer for financial and AI products. Apply the "Day 2 evidence discipline" framework to the following claim.

Claim: [PASTE CLAIM HERE]

For each of the following, give a short, specific assessment (1–3 sentences):

1. **Five pitfalls:** Which of these might apply? Significance vs importance; non-significance vs zero; comparing significant vs non-significant; p-hacking; publication bias. How?

2. **Bias check:** Could survivorship bias, look-ahead bias, or selection bias explain or distort the result?

3. **Baseline check:** What is the relevant naive baseline? Does the claim state whether it beats that baseline?

4. **Overfitting / multiple testing:** Where might trial count, benchmark cherry-picking, or leakage be an issue?

5. **What to ask the vendor:** List 3 concrete questions you would ask before trusting this claim.

Be concise and specific. Do not endorse the claim; focus on what evidence is missing or what could mislead.
```

## Task 5b: Claims to Try (Pick One or More)

**Claim A (backtest):** “Our systematic equity strategy achieved a
Sharpe ratio of 2.1 over 2019–2024.”

**Claim B (LLM/benchmark):** “Our LLM achieves 94% accuracy on
regulatory rule extraction on a benchmark of 36 FCA rules.”

**Claim C (AI product):** “Our AI credit model reduces default rates by
25% compared to traditional scorecards.”

**Claim D (your own):** Paste any real or hypothetical FinTech/AI claim
you want to evaluate.

## Task 5c: Run the Critique and Reflect

1.  Paste one claim into the prompt and run it in your chosen LLM.
2.  Read the model’s response. Note which pitfalls or biases it
    identified and whether it missed any you would add.
3.  *(Optional)* Try the same claim with a different LLM and compare: do
    they emphasise the same gaps?

### Questions (Write 100–150 words)

1.  Did the LLM’s critique align with what you would have asked using
    the Day 2 framework? What did it miss or overstate?
2.  How could you use this prompt (or a shortened version) in your job
    when evaluating a vendor or internal AI/quant claim?

> **No API required**
>
> You can run this exercise in the browser (e.g. ChatGPT, Claude.ai) by
> copying the prompt and claim. In Colab you can optionally use the
> OpenAI or Anthropic API for reproducibility.

The same prompt structure can be used to organise your briefing note
critique in the final assessment.

# Exercise 6 — Tokenisation, Embeddings, and Attention (28 min)

> **Lab B: in-session exercise (15 min in-session; full version here)**
>
> This is **Lab B** from the Day 2 afternoon session. No API key
> required. All code is provided; your job is to run it, change
> parameters, and write short reflections. Tasks 6a–6c take ~5–8 min
> each in Colab; Task 6d is a written reflection connecting the
> observations to the Product D claim. If you are arriving here
> directly, the [setup cell](#setup-colab-only-installs) at the top of
> this notebook installs all required packages.

## Why This Matters

Part III of the lectures described the transformer architecture. This
exercise makes it concrete. You will see *exactly* what the model sees
when it reads a regulatory sentence, why semantically equivalent terms
can have high or low similarity depending on the embedding type, and
where lexical drift originates in the architecture.

The running example from the lectures: *“A Balanced Fund must allocate
at least 30% to FixedIncomeSecurities.”*

------------------------------------------------------------------------

## Task 6a: What Does the Model Actually See? Tokenisation (5 min)

BPE tokenisation splits words into subword units to handle novel or
compound terms. The splits are not linguistic — they are statistical
compressions learned from a general web corpus. Financial compound terms
are particularly susceptible.

In [ ]:
# pip install transformers  (already installed in setup cell)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

financial_terms = [
    "FixedIncomeSecurities",
    "fixed income",
    "Fixed-Income",
    "BondAssets",
    "DebtInstruments",
    "BalancedFund",
    "MortgageBackedSecurities",
    "CounterpartyRisk",
]

print(f"{'Term':<30} {'Tokens':<50} {'Count'}")
print("-" * 85)
for term in financial_terms:
    tokens = tokenizer.tokenize(term)
    print(f"{term:<30} {str(tokens):<50} {len(tokens)}")

> **Try it**
>
> Add your own regulatory or financial terms — try hyphenated terms,
> acronyms (e.g. `UCITS`, `MiFID`, `AUM`), and CamelCase compounds.
> Which patterns produce the most fragmented tokenisation?

### Questions (Write 80–100 words)

1.  How does BERT tokenise `FixedIncomeSecurities` compared to
    `fixed income`? What does this imply for a compliance system that
    needs to match these as equivalent?
2.  A symbolic rule engine doing exact string matching would treat
    `FixedIncomeSecurities` and `BondAssets` as completely different.
    How does tokenisation at least partially close this gap — and where
    does it still fall short?

------------------------------------------------------------------------

## Task 6b: Static vs Contextual Embeddings — The Gap That Matters (10 min)

Static embeddings (Word2Vec, GloVe) assign each word a single fixed
vector regardless of context. Contextual embeddings
(sentence-transformers, BERT) compute the vector from the full sentence.
For financial compliance, where `"bank"` means something different in
`"river bank"` vs `"central bank"`, contextual embeddings are essential.

This task demonstrates the difference using cosine similarity.
Contextual embeddings typically give a similarity well above exact
string match (0) but often moderate (e.g. 0.3–0.5) for surface-form
variants like different casing or spacing; use the heatmap to see where
your run sits.

In [ ]:
# pip install sentence-transformers  (already installed in setup cell)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

model = SentenceTransformer("all-MiniLM-L6-v2")

# Semantically equivalent regulatory phrases — different surface forms
terms = [
    "FixedIncomeSecurities",       # CamelCase compound
    "fixed income securities",     # spaced, lowercase
    "Fixed-Income Securities",     # hyphenated, mixed case
    "BondAssets",                  # synonym, CamelCase
    "debt instruments",            # synonym, spaced
    "government bonds",            # more specific — still similar?
    "BalancedFund",                # different concept — should be lower
    "Equities",                    # clearly different concept
]

embeddings = model.encode(terms)
sim_matrix = cosine_similarity(embeddings)
df_sim = pd.DataFrame(sim_matrix, index=terms, columns=terms)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(df_sim, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_title("Cosine similarity: contextual sentence embeddings\n"
             "(1.0 = identical, 0.0 = orthogonal)", fontsize=11)
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# Print the pairs of greatest interest
print("\nKey pairwise similarities:")
pairs = [
    ("FixedIncomeSecurities", "fixed income securities"),
    ("FixedIncomeSecurities", "BondAssets"),
    ("FixedIncomeSecurities", "Equities"),
    ("BondAssets", "debt instruments"),
]
for a, b in pairs:
    i, j = terms.index(a), terms.index(b)
    print(f"  {a:30s} ↔ {b:30s}  sim = {sim_matrix[i,j]:.3f}")

> **Try it**
>
> Replace `"all-MiniLM-L6-v2"` with `"paraphrase-MiniLM-L6-v2"` and
> re-run. Do the similarity rankings change? Which pairs shift most?
> What does this tell you about model sensitivity to the training
> objective?

### Questions (Write 100–150 words)

1.  `FixedIncomeSecurities` and `fixed income securities` differ only in
    spacing and case. What cosine similarity does the model assign them?
    Is this high or low relative to your expectation?
2.  In your MSc coursework context (or from the lectures), static
    Word2Vec embeddings achieved ~80% classification accuracy while
    contextual BERT embeddings achieved ~98.3%. Looking at the
    similarity heatmap, which property of contextual embeddings explains
    this gap for compound financial terms?
3.  A compliance database using exact string matching would produce a
    similarity of 0 for `FixedIncomeSecurities` vs `fixed income`. The
    embedding model produces a much higher value. Is this always
    beneficial? What failure mode could arise if the similarity is *too*
    high (i.e. the model conflates terms that should be treated as
    legally distinct)?

------------------------------------------------------------------------

## Task 6c: Attention Heatmap — What the Model Attends To (8 min)

Self-attention computes a weighted average over all token
representations, where the weights reflect relevance between pairs of
tokens. Visualising attention weights shows which parts of a regulatory
sentence the model “focuses on” when building representations — and can
reveal whether key quantitative constraints (like “at least 30%”)
receive appropriate attention.

In [ ]:
# BertViz — interactive attention visualisation in Colab
# If this cell throws an error outside Colab, use the browser demo instead (see note below)

try:
    from bertviz import head_view
    from transformers import BertTokenizer, BertModel
    import torch

    bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    bert_model = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
    bert_model.eval()

    sentence = "A Balanced Fund must allocate at least 30 % to FixedIncomeSecurities ."
    inputs = bert_tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = bert_model(**inputs)

    tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    print("Tokens:", tokens)
    print(f"\nModel: {len(outputs.attentions)} layers × "
          f"{outputs.attentions[0].shape[1]} heads")

    # Interactive view (renders in Colab; static fallback below)
    head_view(outputs.attentions, tokens)

except Exception as e:
    print(f"BertViz render error (expected outside Colab): {e}")
    print("Use the browser demo: https://poloclub.github.io/transformer-explainer/")
    print("Paste: 'A Balanced Fund must allocate at least 30% to FixedIncomeSecurities'")

In [ ]:
# Static fallback: plot mean attention from the final layer across all heads
# This runs everywhere, regardless of Colab/BertViz availability

from transformers import BertTokenizer, BertModel
import torch, matplotlib.pyplot as plt, numpy as np

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_model.eval()

sentence = "A Balanced Fund must allocate at least 30 % to FixedIncomeSecurities ."
inputs = bert_tokenizer(sentence, return_tensors="pt")
tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = bert_model(**inputs)

# Average attention across all heads in the last layer
last_layer_attn = outputs.attentions[-1][0]          # shape: (heads, seq, seq)
mean_attn = last_layer_attn.mean(dim=0).numpy()      # shape: (seq, seq)

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(mean_attn, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=8)
ax.set_title("Mean attention weights — final BERT layer (averaged across all heads)\n"
             "Row = query token, Column = key token attended to", fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout(); plt.show()

# Highlight which tokens 'FixedIncomeSecurities' attends to most
fis_tokens = [t for t in tokens if "fixed" in t or "income" in t or "securities" in t or "##" in t]
print(f"\nHow 'FixedIncomeSecurities' was tokenised: {[t for t in tokens if 'fix' in t or '##' in t]}")

> **Try it**
>
> Change the sentence to remove the numerical constraint:
> `"A Balanced Fund must allocate to FixedIncomeSecurities."` Re-run and
> compare the attention heatmap. Does the model attend differently when
> the quantitative constraint (`30%`, `at least`) is absent?

### Questions (Write 100–150 words)

1.  How many tokens does BERT use to represent `FixedIncomeSecurities`?
    What does this tell you about how the model “sees” this compound
    term compared to a human reading it?
2.  Look at the attention heatmap. Does the token for “30” (or “%”)
    receive strong attention from “FixedIncomeSecurities” and
    “allocate”? Why would a compliance system care whether the
    quantitative constraint is linked to the asset class in the
    attention weights?
3.  Attention weights are sometimes interpreted as a measure of “what
    the model focuses on.” What is the limitation of this
    interpretation? (Hint: think about what attention weights actually
    compute, and whether high attention weight implies accurate
    *understanding* of the regulatory intent.)

------------------------------------------------------------------------

## Task 6d: Connecting to the Claim — Architectural Reflection (5 min)

No code required. This task asks you to connect what you have just
observed to the evidence discipline from Part II and the Product D claim
from the lecture discussion.

### Questions (Write 150–200 words)

Recall the claim from the lecture: *“Our LLM achieves 94% accuracy on
regulatory rule extraction (benchmark: 36 FCA rules from the EFC
sourcebook).”*

Using what you have just seen in Tasks 6a–6c, address the following:

1.  **Tokenisation lens:** If the benchmark rules use the term
    `FixedIncomeSecurities` and the model’s training data uses
    `fixed income securities`, what does Task 6a suggest about whether
    the tokeniser treats these as equivalent? Could tokenisation
    differences alone explain some of the 6% error rate?

2.  **Embedding lens:** Task 6b showed that contextual embeddings assign
    high cosine similarity to semantically equivalent forms. Does this
    mean the model will *extract* the rule correctly? What is the
    difference between semantic similarity and structural/legal
    equivalence?

3.  **Attention lens:** The model must not only recognise the asset
    class but also link it to the quantitative constraint
    (`at least 30%`). What does Task 6c suggest about whether attention
    reliably captures this kind of numerical binding?

4.  **Your verdict:** Given these three architectural observations, is
    94% accuracy on 36 rules a strong or weak benchmark? What additional
    evidence would you ask for?

> **Connection to Day 3**
>
> Tomorrow’s session presents real research findings on LLM performance
> on regulatory text. The architectural predictions you have just made —
> about tokenisation fragmentation, embedding–syntax gaps, and attention
> as a noisy proxy for understanding — will either be confirmed or
> challenged by the actual data.

------------------------------------------------------------------------

# Synthesis

## Connecting the Exercises

| Exercise | Key lesson | Day 2 framework |
|--------------------|----------------------|-------------------------------|
| 1 (Bias-variance) | Complexity ≠ accuracy; noise dominates in finance | Overfitting is the default |
| 2 (Survivorship) | Missing data silently distorts results | Failure modes are invisible |
| 3 (Walk-forward) | Full-sample ≠ out-of-sample; baselines matter | Honest evaluation requires discipline |
| 4 (Assessment) | Claims need scrutiny using structured frameworks | Evidence quality is a skill |
| 5 (LLM bridge) | Same discipline for backtests and LLM/benchmark claims | Use the framework interactively with an LLM as critic |
| 6 (Tokenisation, embeddings, attention) | Architecture creates failure modes before the model even “understands” the text | Lexical drift and attention limits are structural, not incidental |

## Final Reflection (150–200 words)

Considering all exercises, write a short reflection:

1.  Before today, how would you have evaluated a quantitative claim from
    a FinTech vendor? What would you do differently now?
2.  Which failure mode (overfitting, survivorship, look-ahead, multiple
    testing) do you think is most commonly exploited in financial
    marketing? Why?
3.  Exercise 6 showed that architectural limitations — tokenisation
    fragmentation, embedding–syntax gaps, attention as a noisy proxy —
    exist before any fine-tuning or deployment decisions are made. Does
    this change how you would interpret a vendor’s accuracy benchmark?
    What is the minimum architectural information you would now demand
    alongside any reported accuracy figure?

## References